# Thermal Analysis Simulation

This notebook defines a 2D thermal diffusion simulation using sim2l.

It demonstrates:
- Defining inputs with `%%sim2l_inputs`
- Defining outputs with `%%sim2l_outputs`
- Simulation logic
- Saving outputs with `save_outputs()`
- Deploying the simulation

In [ ]:
# Load sim2l IPython extension
%load_ext sim2l.notebook

import sim2l
print(f"sim2l version: {sim2l.__version__}")
print("✓ sim2l magics loaded")

## 1. Define Inputs

Use the `%%sim2l_inputs` magic to define input parameters in YAML format.

In [ ]:
%%sim2l_inputs

temperature:
  type: Number
  units: kelvin
  min: 0
  max: 1000
  default: 300
  description: "Initial temperature of the system"

power:
  type: Number
  units: watt
  min: 0
  max: 100
  default: 10
  description: "Applied power"

iterations:
  type: Integer
  min: 1
  max: 10000
  default: 100
  description: "Number of simulation iterations"

grid_size:
  type: Integer
  min: 10
  max: 200
  default: 50
  description: "Size of the simulation grid (NxN)"

## 2. Define Outputs

Use the `%%sim2l_outputs` magic to define expected outputs.

In [ ]:
%%sim2l_outputs

max_temperature:
  type: Number
  units: kelvin
  description: "Maximum temperature reached in the system"

min_temperature:
  type: Number
  units: kelvin
  description: "Minimum temperature in the system"

avg_temperature:
  type: Number
  units: kelvin
  description: "Average temperature across the grid"

temperature_distribution:
  type: Array
  dtype: float
  description: "Final temperature distribution (flattened)"

thermal_plot:
  type: Image
  description: "Visualization of temperature distribution"

converged:
  type: Boolean
  description: "Whether the simulation converged"

iterations_to_convergence:
  type: Integer
  description: "Number of iterations until convergence"

## 3. Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import sim2l

print(f"sim2l version: {sim2l.__version__}")

## 4. Load Input Parameters

For interactive testing, load the input schema and set test values.

In [ ]:
# Get inputs
# When executed by Papermill, parameters are injected as variables
# When run interactively, use get_inputs() with defaults

try:
    # Check if parameters were injected by Papermill
    # If temperature variable exists, we're in Papermill mode
    _ = temperature
    print("Using Papermill-injected parameters")
except NameError:
    # Interactive mode - get inputs from schema
    import sim2l
    inputs = sim2l.get_inputs()
    
    # Extract values for testing
    temperature = 300
    power = 20
    iterations = 500
    grid_size = 50
    
    print(f"Using default test values:")
    print(f"  Temperature: {temperature} K")
    print(f"  Power: {power} W")
    print(f"  Iterations: {iterations}")
    print(f"  Grid size: {grid_size}")


## 5. Simulation Logic

2D thermal diffusion using finite difference method.

In [ ]:
# Use parameter values directly
T_initial = temperature
power_applied = power
max_iterations = iterations
n = grid_size

print(f"\nStarting simulation with:")
print(f"  Initial temperature: {T_initial} K")
print(f"  Applied power: {power_applied} W")
print(f"  Max iterations: {max_iterations}")
print(f"  Grid size: {n}x{n}")

# Initialize temperature field
T = np.ones((n, n)) * T_initial

# Simulation parameters
alpha = 0.01      # thermal diffusivity
dx = 1.0          # spatial step
dt = 0.01         # time step
convergence_threshold = 1e-3

# Heat source (applied at center)
heat_source = power_applied / (4 * 4)  # Distribute over 4x4 center region

# Track convergence
converged = False
iterations_to_convergence = max_iterations

print("\nRunning thermal diffusion simulation...")

# Main simulation loop
for iteration in range(max_iterations):
    T_old = T.copy()
    
    # Apply 2D heat equation (finite difference)
    # ∂T/∂t = α * (∂²T/∂x² + ∂²T/∂y²)
    T[1:-1, 1:-1] = T_old[1:-1, 1:-1] + alpha * dt / dx**2 * (
        T_old[2:, 1:-1] + T_old[:-2, 1:-1] +
        T_old[1:-1, 2:] + T_old[1:-1, :-2] -
        4 * T_old[1:-1, 1:-1]
    )
    
    # Apply heat source at center
    center_x = n // 2
    center_y = n // 2
    T[center_x-2:center_x+2, center_y-2:center_y+2] += heat_source * dt
    
    # Check for convergence
    max_change = np.max(np.abs(T - T_old))
    
    if max_change < convergence_threshold:
        converged = True
        iterations_to_convergence = iteration + 1
        print(f"Converged at iteration {iteration + 1}")
        print(f"  Max change: {max_change:.6f}")
        break
    
    # Progress update every 100 iterations
    if (iteration + 1) % 100 == 0:
        print(f"  Iteration {iteration + 1}/{max_iterations}, max change: {max_change:.6f}")

if not converged:
    print(f"Did not converge after {max_iterations} iterations")

print("\nSimulation complete!")


## 6. Compute Results

In [ ]:
# Calculate statistics
max_temp = np.max(T)
min_temp = np.min(T)
avg_temp = np.mean(T)

print("Results:")
print(f"  Max Temperature: {max_temp:.2f} K")
print(f"  Min Temperature: {min_temp:.2f} K")
print(f"  Avg Temperature: {avg_temp:.2f} K")
print(f"  Converged: {converged}")
print(f"  Iterations to Convergence: {iterations_to_convergence}")

## 7. Generate Visualization

In [ ]:
# Create visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# 2D heatmap
im1 = ax1.imshow(T, cmap='hot', interpolation='bilinear')
ax1.set_title(f'Temperature Distribution\nMax: {max_temp:.1f} K', fontsize=12)
ax1.set_xlabel('X position')
ax1.set_ylabel('Y position')
plt.colorbar(im1, ax=ax1, label='Temperature (K)')

# 3D surface plot
from mpl_toolkits.mplot3d import Axes3D
ax2 = fig.add_subplot(122, projection='3d')
x = np.arange(n)
y = np.arange(n)
X, Y = np.meshgrid(x, y)
surf = ax2.plot_surface(X, Y, T, cmap='hot', linewidth=0, antialiased=True)
ax2.set_title('3D Temperature Profile', fontsize=12)
ax2.set_xlabel('X')
ax2.set_ylabel('Y')
ax2.set_zlabel('Temperature (K)')
fig.colorbar(surf, ax=ax2, shrink=0.5, aspect=5, label='Temperature (K)')

plt.tight_layout()

# Save plot
plot_filename = 'thermal_plot.png'
plt.savefig(plot_filename, dpi=150, bbox_inches='tight')
print(f"\nPlot saved to: {plot_filename}")

plt.show()

## 8. Save Outputs

Use `sim2l.save_outputs()` to save all outputs according to the schema.

In [ ]:
# Save outputs
sim2l.save_outputs(
    max_temperature=float(max_temp),
    min_temperature=float(min_temp),
    avg_temperature=float(avg_temp),
    temperature_distribution=T.flatten(),
    thermal_plot=plot_filename,
    converged=bool(converged),
    iterations_to_convergence=int(iterations_to_convergence)
)

print("\n✓ Outputs saved successfully!")

## 9. Deploy Simulation (Optional)

After testing, deploy this notebook as a versioned simulation.

In [ ]:
# Deploy this notebook as a simulation
# Uncomment to deploy:

# sim2l.deploy_simulation(
#     notebook="thermal_simulation.ipynb",
#     name="thermal_analysis",
#     version="1.0.0",
#     description="2D thermal diffusion simulation with finite difference method",
#     author="Your Name",
#     tags=["physics", "thermal", "finite-difference", "diffusion"]
# )
# 
# print("✓ Simulation deployed to database!")
# print("  Now you can execute it from anywhere:")
# print("  >>> sim = sim2l.load_simulation('thermal_analysis')")
# print("  >>> result = sim.run(temperature=350, power=20, iterations=500)")

## Summary

This notebook demonstrates:

1. **Input Definition**: Using `%%sim2l_inputs` magic to define typed, validated inputs with units
2. **Output Definition**: Using `%%sim2l_outputs` magic to define expected outputs
3. **Simulation Logic**: Standard Python code for the simulation
4. **Output Saving**: Using `sim2l.save_outputs()` to save results according to schema
5. **Deployment**: Optional deployment to database for reuse

### Key Features:
- Type-safe inputs with validation (min/max, units)
- Automatic unit handling (kelvin, watt)
- Structured outputs (scalars, arrays, images)
- Can be deployed and executed from anywhere

### Next Steps:
1. Test interactively with different parameters
2. Deploy to database
3. Execute from other notebooks/scripts
4. Run parameter sweeps
5. Use caching for repeated executions